# Preparación de datos demográficos de OWID (2000–2022)
# Nombre: Juan Carlos Orozco

**Objetivo:** descargar cuatro conjuntos de datos, revisar sus columnas, preparar cada uno por separado y unirlos por `Entity` y `Year`. El resultado servirá para una práctica posterior de aprendizaje automático.

Ejecuta las celdas en orden. Requiere `pandas` y `matplotlib`, además de acceso a internet. Fuente: [Our World in Data](https://ourworldindata.org/).

In [0]:
import pandas as pd
import matplotlib.pyplot as plt

OPCIONES_OWID = {"User-Agent": "Our World In Data data fetch/1.0"}
BASE = "https://ourworldindata.org/grapher/"
SUFIJO = ".csv?v=1&csvType=full&useColumnShortNames=true"

## 1. Descargar los cuatro archivos

Las direcciones son cadenas de texto simples. Cada archivo se carga en su propio DataFrame.

In [0]:
url_population = BASE + "population-with-un-projections" + SUFIJO
url_births = BASE + "number-of-births-per-year" + SUFIJO
url_deaths = BASE + "number-of-deaths-per-year" + SUFIJO
url_child_deaths = BASE + "number-of-child-deaths-unwpp" + SUFIJO

population = pd.read_csv(url_population, storage_options=OPCIONES_OWID)
births = pd.read_csv(url_births, storage_options=OPCIONES_OWID)
deaths = pd.read_csv(url_deaths, storage_options=OPCIONES_OWID)
child_deaths = pd.read_csv(url_child_deaths, storage_options=OPCIONES_OWID)

print("Filas y columnas:")
print("population:", population.shape)
print("births:", births.shape)
print("deaths:", deaths.shape)
print("child_deaths:", child_deaths.shape)

## 2. Explorar las columnas

OWID puede modificar los nombres de los indicadores. Revisamos las columnas reales antes de seleccionarlas.

In [0]:
for nombre, tabla in [("population", population), 
                      ("births", births),
                      ("deaths", deaths), 
                      ("child_deaths", child_deaths)]:
    print(f"\n{nombre}: {tabla.columns.tolist()}")
    display(tabla.head(3))

## 3. Preparar población

Elegimos la columna de **estimaciones históricas** para ambos sexos y todas las edades; excluimos las proyecciones. Después conservamos los años de interés.

In [0]:
columnas_population = [
    columna for columna in population.columns
        if "population" in columna and "estimates" in columna]

print("Columnas candidatas:", columnas_population)

assert len(columnas_population) == 1, "Revisar manualmente el indicador de población mostrado arriba"


col_population = columnas_population[0]

population_limpio = population[["entity", "year", col_population]].copy()

population_limpio = population_limpio.rename(columns={col_population: "population"})

population_limpio = population_limpio[population_limpio["year"].between(2000, 2022)].copy()

print(population_limpio.shape)
display(population_limpio.head())

## 4. Preparar nacimientos

Seleccionamos el número anual de nacimientos históricos. Revisamos qué entidades comparte con las otras tablas.

In [0]:
columnas_births = [c for c in births.columns
                  if "births" in c and "estimates" in c]


print("Columnas candidatas:", columnas_births)
assert len(columnas_births) == 1, "Revisar manualmente el indicador de nacimientos mostrado arriba"
col_births = columnas_births[0]

births_limpio = births[["entity", "year", col_births]].copy()
births_limpio = births_limpio.rename(columns={col_births: "births"})
births_limpio = births_limpio[births_limpio["year"].between(2000, 2022)].copy()
print(births_limpio.shape)
display(births_limpio.head())

## 5. Preparar defunciones totales

In [0]:
columnas_deaths = [c for c in deaths.columns
                   if "deaths" in c and "estimates" in c]

print("Columnas candidatas:", columnas_deaths)
assert len(columnas_deaths) == 1, "Revisar manualmente el indicador de defunciones mostrado arriba"

col_deaths = columnas_deaths[0]

deaths_limpio = deaths[["entity", "year", col_deaths]].copy()
deaths_limpio = deaths_limpio.rename(columns={col_deaths: "deaths"})
deaths_limpio = deaths_limpio[deaths_limpio["year"].between(2000, 2022)].copy()

print(deaths_limpio.shape)
display(deaths_limpio.head())

## 6. Preparar defunciones de menores de cinco años

Este indicador corresponde a niños de 0 a 4 años; no debe confundirse con mortalidad de menores de un año.

In [0]:
columnas_child = [c for c in child_deaths.columns
                  if c not in ["entity", "code", "year"]]

print("Columnas candidatas:", columnas_child)

assert len(columnas_child) == 1, "Revisar manualmente el indicador de menores de cinco años mostrado arriba"

col_child = columnas_child[0]

child_deaths_limpio = child_deaths[["entity", "year", col_child]].copy()
child_deaths_limpio = child_deaths_limpio.rename(columns={col_child: "child_deaths_under_5"})

child_deaths_limpio = child_deaths_limpio[child_deaths_limpio["year"].between(2000, 2022)].copy()

print(child_deaths_limpio.shape)
display(child_deaths_limpio.head())

## 7. Revisar llaves y cobertura

Cada combinación `Entity`–`Year` debe aparecer una sola vez en cada tabla. `Code` no se utiliza como llave porque puede faltar en agregados regionales.

In [0]:
tablas = [("population", population_limpio), ("births", births_limpio),
          ("deaths", deaths_limpio), ("child_deaths", child_deaths_limpio)]

for nombre, tabla in tablas:
    duplicados = tabla.duplicated(["entity", "year"]).sum()
    print(nombre, "| entidades:", tabla["entity"].nunique(),
          "| años:", tabla["year"].min(), "a", tabla["year"].max(),
          "| llaves duplicadas:", duplicados)
    assert duplicados == 0, f"Llaves duplicadas en {nombre}"

entidades_comunes = set(population_limpio["entity"]) & set(births_limpio["entity"]) \
                  & set(deaths_limpio["entity"]) & set(child_deaths_limpio["entity"])

print("Entidades presentes en los cuatro archivos:", len(entidades_comunes))
print(sorted(entidades_comunes)[:30])

## 8. Unir las tablas, un paso a la vez

Usamos `inner`: permanecen únicamente las combinaciones de entidad y año presentes en ambos DataFrames. `validate="one_to_one"` detecta llaves duplicadas.

In [0]:
population_births = pd.merge(
    population_limpio, births_limpio,
    on=["entity", "year"], how="inner", validate="one_to_one"
)
print("Después de población + nacimientos:", population_births.shape)
display(population_births.head())

In [0]:
population_births_deaths = pd.merge(
    population_births, deaths_limpio,
    on=["entity", "year"], how="inner", validate="one_to_one"
)
print("Después de agregar defunciones:", population_births_deaths.shape)
display(population_births_deaths.head())

In [0]:
demografia = pd.merge(
    population_births_deaths, child_deaths_limpio,
    on=["entity", "year"], how="inner", validate="one_to_one"
)
print("Dataset unido:", demografia.shape)
display(demografia.head())

## 9. Revisar faltantes y crear tasas

Eliminamos solo filas con algún indicador necesario ausente. Las tasas de nacimientos y defunciones se expresan por 1,000 habitantes. La razón de muertes de menores de cinco años usa nacimientos del **mismo año** como denominador; es una medida descriptiva aproximada, no una tasa de mortalidad de cohorte.

In [0]:
print("Valores faltantes antes de limpiar:")
print(demografia.isna().sum())
print("Filas antes de limpiar:", len(demografia))

demografia = demografia.dropna(subset=["population", "births", "deaths", "child_deaths_under_5"]).copy()
demografia = demografia[(demografia["population"] > 0) & (demografia["births"] > 0)].copy()

print("Filas después de limpiar:", len(demografia))
print("Entidades después de limpiar:", demografia["entity"].nunique())
assert not demografia.empty, "No hay entidades y años coincidentes; revisar fuentes y columnas"

In [0]:
demografia["birth_rate_per_1000"] = demografia["births"] / demografia["population"] * 1000
demografia["death_rate_per_1000"] = demografia["deaths"] / demografia["population"] * 1000
demografia["under5_deaths_per_1000_births_same_year"] = (
    demografia["child_deaths_under_5"] / demografia["births"] * 1000
)

demografia = demografia[[
    "entity", "year", "population", "births", "deaths", "child_deaths_under_5",
    "birth_rate_per_1000", "death_rate_per_1000",
    "under5_deaths_per_1000_births_same_year"
]].sort_values(["entity", "year"]).reset_index(drop=True)

display(demografia.head())
display(demografia.describe(include="all"))

## 10. Visualizar una entidad disponible

Puedes cambiar `entidad` por cualquier nombre de `demografia["Entity"].unique()`.

In [0]:
entidad = "Mexico" if "Mexico" in demografia["entity"].values else demografia["entity"].iloc[0]
ejemplo = demografia[demografia["entity"] == entidad].sort_values("year")

ejemplo.plot(x="year", y=["birth_rate_per_1000", "death_rate_per_1000"],
             marker="o", figsize=(9, 4), title=f"Tasas demográficas: {entidad}")
plt.ylabel("Por 1,000 habitantes")
plt.grid(alpha=0.3)
plt.show()

## 11. Guardar el resultado

El CSV se guarda junto al notebook. Contiene solamente entidades y años compartidos por las cuatro fuentes.

In [0]:
from pathlib import Path

carpeta_salida = Path("../data/processed")
carpeta_salida.mkdir(parents=True, exist_ok=True)

salida = carpeta_salida / "demografia_owid_2000_2022.csv"
demografia.to_csv(salida, index=False)

print(f"Archivo guardado: {salida.resolve()} | filas: {len(demografia)}")

## 12. Actividad: Completar el dataset con Gasto en Salud, PIB y Esperanza de Vida

**Objetivo:** Extender el dataset demográfico construido en clase (`demografia`) incorporando tres nuevos indicadores obtenidos de *Our World in Data*:
1. Gasto corriente en salud como porcentaje del PIB (`health_percentage`).
2. PIB per cápita ajustado por PPA (`gdp`).
3. Esperanza de vida al nacer (`life_expectancy`).

El resultado final deberá contener exactamente las siguientes **9 columnas** (7 variables de interés + 2 llaves):
- `'entity'`
- `'year'`
- `'population'`
- `'births'`
- `'deaths'`
- `'child_deaths_under_5'`
- `'health_percentage'`
- `'gdp'`
- `'life_expectancy'`

### 12.1. Descargar gasto en salud (% del PIB)

In [0]:
import pandas as pd
import requests

# Fetch the data.
health = pd.read_csv("https://ourworldindata.org/grapher/total-healthcare-expenditure-gdp.csv?v=1&csvType=full&useColumnShortNames=true", storage_options = {'User-Agent': 'Our World In Data data fetch/1.0'})
df = health  # Alias para compatibilidad

# Fetch the metadata
metadata_health = requests.get("https://ourworldindata.org/grapher/total-healthcare-expenditure-gdp.metadata.json?v=1&csvType=full&useColumnShortNames=true").json()
metadata = metadata_health

print("Dimensiones health:", health.shape)
display(health.head())

### 12.2. Descargar PIB per cápita

In [0]:
import pandas as pd
import requests

# Fetch the data.
gdp = pd.read_csv("https://ourworldindata.org/grapher/gdp-per-capita-worldbank.csv?v=1&csvType=full&useColumnShortNames=true", storage_options = {'User-Agent': 'Our World In Data data fetch/1.0'})
df = gdp  # Alias para compatibilidad

# Fetch the metadata
metadata_gdp = requests.get("https://ourworldindata.org/grapher/gdp-per-capita-worldbank.metadata.json?v=1&csvType=full&useColumnShortNames=true").json()
metadata = metadata_gdp

print("Dimensiones gdp:", gdp.shape)
display(gdp.head())

### 12.3. Descargar esperanza de vida

In [0]:
import pandas as pd

# Fetch the data.
df_life = pd.read_csv("https://ourworldindata.org/grapher/life-expectancy.csv?v=1&csvType=full&useColumnShortNames=true", storage_options = {'User-Agent': 'Our World In Data data fetch/1.0'})

print("Dimensiones df_life:", df_life.shape)
display(df_life.head())

### 12.4. Preparar las columnas de los tres indicadores

Seleccionamos los indicadores de interés, renombramos a `health_percentage`, `gdp` y `life_expectancy`, y conservamos los años 2000 a 2022.

In [0]:
# 1. Preparar gasto en salud (% del PIB)
columnas_health = [c for c in health.columns if c not in ["entity", "code", "year"]]
print("Columna identificada para salud:", columnas_health)
assert len(columnas_health) == 1, "Revisar columnas de salud"
col_health = columnas_health[0]

health_limpio = health[["entity", "year", col_health]].copy()
health_limpio = health_limpio.rename(columns={col_health: "health_percentage"})
health_limpio = health_limpio[health_limpio["year"].between(2000, 2022)].copy()

# 2. Preparar PIB per cápita
columnas_gdp = [c for c in gdp.columns if c not in ["entity", "code", "year", "owid_region"]]
print("Columna identificada para PIB:", columnas_gdp)
assert len(columnas_gdp) == 1, "Revisar columnas de PIB"
col_gdp = columnas_gdp[0]

gdp_limpio = gdp[["entity", "year", col_gdp]].copy()
gdp_limpio = gdp_limpio.rename(columns={col_gdp: "gdp"})
gdp_limpio = gdp_limpio[gdp_limpio["year"].between(2000, 2022)].copy()

# 3. Preparar esperanza de vida
columnas_life = [c for c in df_life.columns if c not in ["entity", "code", "year"]]
print("Columna identificada para esperanza de vida:", columnas_life)
assert len(columnas_life) == 1, "Revisar columnas de esperanza de vida"
col_life = columnas_life[0]

life_limpio = df_life[["entity", "year", col_life]].copy()
life_limpio = life_limpio.rename(columns={col_life: "life_expectancy"})
life_limpio = life_limpio[life_limpio["year"].between(2000, 2022)].copy()

print("Dimensiones health_limpio:", health_limpio.shape)
print("Dimensiones gdp_limpio:", gdp_limpio.shape)
print("Dimensiones life_limpio:", life_limpio.shape)
display(health_limpio.head())
display(gdp_limpio.head())
display(life_limpio.head())

### 12.5. Revisar llaves y duplicados

In [0]:
tablas_nuevas = [
    ("health_percentage", health_limpio), 
    ("gdp", gdp_limpio), 
    ("life_expectancy", life_limpio)
]

for nombre, tabla in tablas_nuevas:
    duplicados = tabla.duplicated(["entity", "year"]).sum()
    print(nombre, "| entidades:", tabla["entity"].nunique(),
          "| años:", tabla["year"].min(), "a", tabla["year"].max(),
          "| llaves duplicadas:", duplicados)
    assert duplicados == 0, f"Llaves duplicadas en {nombre}" 

### 12.6. Unir con el dataset demográfico de clase y generar `df_7variables`

Unimos con `demografia` mediante `inner join`, validando relación `one_to_one`. Luego conservamos exactamente las 9 columnas requeridas:
`['entity', 'year', 'population', 'births', 'deaths', 'child_deaths_under_5', 'health_percentage', 'gdp', 'life_expectancy']`

In [0]:
# Unir el dataset demográfico de clase con salud, PIB y esperanza de vida
demografia_completo = pd.merge(
    demografia, health_limpio,
    on=["entity", "year"], how="inner", validate="one_to_one"
)

demografia_completo = pd.merge(
    demografia_completo, gdp_limpio,
    on=["entity", "year"], how="inner", validate="one_to_one"
)

demografia_completo = pd.merge(
    demografia_completo, life_limpio,
    on=["entity", "year"], how="inner", validate="one_to_one"
)

# Definir exactamente las 9 columnas requeridas (7 variables + entity y year)
columnas_salida = [
    'entity', 
    'year', 
    'population', 
    'births', 
    'deaths', 
    'child_deaths_under_5', 
    'health_percentage', 
    'gdp', 
    'life_expectancy'
]

# Seleccionar columnas en el orden especificado y ordenar por entidad y año
df_7variables = demografia_completo[columnas_salida].sort_values(["entity", "year"]).reset_index(drop=True)

# Limpiar posibles filas con valores faltantes
df_7variables = df_7variables.dropna(subset=columnas_salida).copy()

# Variables disponibles para compatibilidad
dataset_final = df_7variables
demografia_final = df_7variables
df = df_7variables

# Verificaciones
assert list(df_7variables.columns) == columnas_salida, f"Error: las columnas deben ser exactamente {columnas_salida}"
assert not df_7variables.empty, "El dataset final está vacío"

print("Dimensiones de df_7variables:", df_7variables.shape)
print("Número de entidades únicas:", df_7variables["entity"].nunique())
print("Rango de años:", df_7variables["year"].min(), "-", df_7variables["year"].max())
print("\nColumnas del dataset final:")
for col in df_7variables.columns:
    print(f" - '{col}'")

display(df_7variables.head(10))
display(df_7variables.describe())

### 12.7. Guardar el dataset en data/raw

Guardamos el dataset final `df_7variables` en la carpeta `data/raw`.

In [0]:
# Guardar el df completo
from pathlib import Path
carpeta_salida = Path("../data/raw")
carpeta_salida.mkdir(parents=True, exist_ok=True)
salida = carpeta_salida / "dataset_7variables_2000_2022.csv"
df_7variables.to_csv(salida, index=False)
print(f"Archivo guardado: {salida.resolve()} \n | filas: {len(df_7variables)}")

# También guardar en data/raw dentro de la misma práctica
carpeta_local = Path("data/raw")
carpeta_local.mkdir(parents=True, exist_ok=True)
salida_local = carpeta_local / "dataset_7variables_2000_2022.csv"
df_7variables.to_csv(salida_local, index=False)
print(f"Archivo guardado (práctica local): {salida_local.resolve()}")